# Reasoning Explorer

Per-startup reasoning drilldown across all ablation conditions (Single, Multi-Analyst, TextGrad).

**Sections:**
1. Startup Browser — side-by-side reasoning across conditions
2. Analyst Vote Patterns — agreement, optimism bias, confidence calibration
3. Judge Dimension Scores — radar chart + per-startup heatmap
4. Misclassification Analysis — FP/FN autopsy by sector
5. Sector Performance — AUROC/AP@K by category (requires re-run after B2 fix)
6. TextGrad Gradient Analysis (requires re-run after B1 fix)

Reload from top after any new experiment run.

In [2]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available — startup browser will use static table")

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.titlesize"] = 12

PROJECT_ROOT = Path("..").resolve()
ABL_DIR      = PROJECT_ROOT / "results" / "ablation"
JUDGE_DIR    = PROJECT_ROOT / "results" / "judge_evaluation"
TG_DIR       = PROJECT_ROOT / "results" / "textgrad_validation"

CONDITIONS    = ["single", "multi", "textgrad"]
ANALYST_NAMES = ["Market", "Business Model", "Feasibility", "Team"]
JUDGE_DIMS    = ["product_novelty", "market_opportunity", "feasibility",
                 "team_quality", "reasoning_coherence", "risk_identification"]
JUDGE_LABELS  = ["Product\nNovelty", "Market\nOpp.", "Feasibility",
                 "Team\nQuality", "Reasoning\nCoherence", "Risk\nID"]
COLORS = {"single": "#3498db", "multi": "#2ecc71", "textgrad": "#e67e22"}

In [3]:
def load_predictions(condition: str) -> pd.DataFrame | None:
    """Load most recent predictions JSONL for a condition, checking archive if needed."""
    files = sorted(ABL_DIR.glob(f"{condition}_*_predictions.jsonl"))
    if not files:
        archive = ABL_DIR / "archive"
        if archive.exists():
            for sub in sorted(archive.iterdir(), reverse=True):
                files = sorted(sub.glob(f"{condition}_*_predictions.jsonl"))
                if files:
                    break
    if not files:
        return None
    rows = [json.loads(ln) for ln in files[-1].read_text().splitlines() if ln.strip()]
    df = pd.DataFrame(rows)
    df["condition"] = condition
    return df


def load_judge_scores() -> pd.DataFrame | None:
    files = sorted(JUDGE_DIR.glob("judge_scores_*.jsonl"), key=lambda p: p.stat().st_mtime)
    if not files:
        return None
    rows = [json.loads(ln) for ln in files[-1].read_text().splitlines() if ln.strip()]
    return pd.DataFrame(rows)


def load_sector_metrics(condition: str) -> dict | None:
    files = sorted(ABL_DIR.glob(f"{condition}_*_sector_metrics.json"))
    if not files:
        archive = ABL_DIR / "archive"
        if archive.exists():
            for sub in sorted(archive.iterdir(), reverse=True):
                files = sorted(sub.glob(f"{condition}_*_sector_metrics.json"))
                if files:
                    break
    if not files:
        return None
    return json.loads(files[-1].read_text())


Predictions loaded:
  single       n=30
  multi        n=30
  textgrad     n=30

Judge scores: n=39


---
## Section 1: Startup Browser

Select any startup to compare decisions and reasoning across all three conditions side-by-side.

**Answers:** What does each model actually say about a specific startup, and do they agree?

In [4]:
# Build index of all evaluated startups
all_startups: dict[str, str] = {}
for df in dfs.values():
    if df is not None:
        for _, row in df.iterrows():
            all_startups.setdefault(row["object_id"], row.get("name", row["object_id"]))

startup_options = sorted(all_startups.items(), key=lambda x: x[1])

def show_startup(object_id: str) -> None:
    name = all_startups.get(object_id, object_id)
    print(f"\n{'='*80}")
    print(f"  {name}  [{object_id}]")
    print(f"{'='*80}")
    for c, df in dfs.items():
        if df is None:
            print(f"  [{c.upper():10}] not evaluated")
            continue
        row = df[df["object_id"] == object_id]
        if row.empty:
            print(f"  [{c.upper():10}] not in this split")
            continue
        r = row.iloc[0]
        prob = r.get("probability_float")
        prob_str = f"{prob:.0%}" if prob is not None else "?"
        correct = (r["decision"] == "INVEST") == (r["target"] == 1)
        gt = "INVEST" if r["target"] == 1 else "PASS"
        print(f"\n  [{c.upper():10}]  Decision: {str(r.get('decision','?')):5}  "
              f"Prob: {prob_str}  GT: {gt}  {'✓ CORRECT' if correct else '✗ WRONG'}")
        print(f"  Reasoning: {str(r.get('reasoning', ''))[:300]}")
        aa = r.get("analyst_assessments")
        if isinstance(aa, dict):
            print("  Analyst votes:")
            for analyst in ANALYST_NAMES:
                a = aa.get(analyst, {})
                dec = a.get("decision", "?")
                conf = a.get("confidence", "?")
                print(f"    {analyst:<20} {dec:<15} conf={conf}")

if WIDGETS_AVAILABLE and startup_options:
    dropdown = widgets.Dropdown(
        options=[(f"{name}  [{oid}]", oid) for oid, name in startup_options],
        description="Startup:",
        layout=widgets.Layout(width="65%"),
    )
    out = widgets.Output()

    def _on_change(change):
        with out:
            out.clear_output()
            show_startup(change["new"])

    dropdown.observe(_on_change, names="value")
    display(widgets.VBox([dropdown, out]))
    # Trigger initial display
    show_startup(startup_options[0][0])
else:
    # Static fallback: print all startups as a table
    rows = []
    for oid, name in startup_options:
        row = {"object_id": oid, "name": name}
        for c, df in dfs.items():
            if df is not None:
                match = df[df["object_id"] == oid]
                row[c] = match.iloc[0]["decision"] if not match.empty else "-"
        rows.append(row)
    display(pd.DataFrame(rows))


  Agios Pharmaceuticals  [c:38752]

  [SINGLE    ]  Decision: INVEST  Prob: 80%  GT: INVEST  ✓ CORRECT
  Reasoning: The startup demonstrates strong textual description quality, operates in a high-growth sector with strong exit potential, has a credentialed team, and has received significant funding. These factors collectively suggest a high probability of successful exit.

  [MULTI     ]  Decision: INVEST  Prob: 90%  GT: INVEST  ✓ CORRECT
  Reasoning: All four analysts have high confidence in the company's promising market position, robust business model, strong product viability, and capable team composition. The consistent 'PROMISING' assessments across all categories indicate a high likelihood of successful exit.
  Analyst votes:
    Market               PROMISING       conf=85
    Business Model       PROMISING       conf=85
    Feasibility          PROMISING       conf=85
    Team                 PROMISING       conf=85

  [TEXTGRAD  ]  Decision: INVEST  Prob: 90%  GT: INVEST  ✓ 

---
## Section 2: Analyst Vote Patterns

Applies to multi-analyst and TextGrad conditions only.

**Answers:**
- Do analysts act as independent signals, or are their votes correlated?
- Which analyst is most optimistic?
- Does high analyst confidence actually predict startup success?

In [ ]:
vote_dfs = {
    c: extract_analyst_votes(dfs[c])
    for c in ["multi", "textgrad"]
    if dfs.get(c) is not None
}

if not vote_dfs:
    print("No multi-analyst predictions found.")
else:
    # Analyst vote correlation heatmap
    fig, axes = plt.subplots(1, len(vote_dfs), figsize=(6 * len(vote_dfs), 5))
    if len(vote_dfs) == 1:
        axes = [axes]
    for ax, (cond, vdf) in zip(axes, vote_dfs.items()):
        vote_cols = [f"{a}_vote" for a in ANALYST_NAMES]
        corr = vdf[vote_cols].corr()
        corr.index = ANALYST_NAMES
        corr.columns = ANALYST_NAMES
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
                    vmin=-1, vmax=1, ax=ax, linewidths=0.5)
        ax.set_title(f"[{cond.upper()}] Analyst Vote Correlation\n(1=PROMISING, 0=NOT_PROMISING)")
    plt.tight_layout()
    plt.show()

In [ ]:
if vote_dfs:
    # Analyst optimism: fraction voting PROMISING
    fig, axes = plt.subplots(1, len(vote_dfs), figsize=(7 * len(vote_dfs), 4))
    if len(vote_dfs) == 1:
        axes = [axes]
    for ax, (cond, vdf) in zip(axes, vote_dfs.items()):
        vote_cols = [f"{a}_vote" for a in ANALYST_NAMES]
        means = vdf[vote_cols].mean()
        bar_colors = ["#3498db", "#2ecc71", "#e67e22", "#9b59b6"]
        bars = ax.bar(ANALYST_NAMES, means.values, color=bar_colors, alpha=0.85)
        ax.set_ylim(0, 1.1)
        ax.set_ylabel("Fraction voting PROMISING")
        ax.set_title(f"[{cond.upper()}] Analyst Optimism")
        ax.axhline(0.5, ls="--", color="gray", alpha=0.6, label="Neutral")
        ax.legend()
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.02,
                    f"{h:.2f}", ha="center", va="bottom", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
if vote_dfs:
    # Analyst confidence calibration: does high confidence predict success?
    # Note: with small n, treat this as exploratory rather than definitive.
    fig, axes = plt.subplots(1, len(ANALYST_NAMES), figsize=(14, 4), sharey=True)
    for ax, analyst in zip(axes, ANALYST_NAMES):
        for cond, vdf in vote_dfs.items():
            conf_col = f"{analyst}_conf"
            if conf_col not in vdf.columns:
                continue
            sub = vdf[[conf_col, "target"]].dropna()
            if len(sub) < 4:
                continue
            try:
                sub = sub.copy()
                sub["conf_bin"] = pd.qcut(sub[conf_col], q=4, duplicates="drop")
                hit = sub.groupby("conf_bin", observed=True)["target"].mean()
                ax.plot(range(len(hit)), hit.values, marker="o",
                        color=COLORS.get(cond, "gray"), label=cond, linewidth=1.5)
            except Exception:
                pass
        ax.set_title(analyst, fontsize=10)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("Confidence quartile (low→high)")
    axes[0].set_ylabel("Actual success rate")
    axes[0].legend(fontsize=8)
    fig.suptitle("Analyst Confidence vs. Actual Success Rate\n"
                 "(upward slope = confidence is a useful signal)", fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## Section 3: Judge Dimension Scores

LLM-as-judge rubric scores on 6 dimensions (1–5 scale) per condition.

**Answers:** Does TextGrad improve reasoning quality on specific dimensions even when quantitative metrics are similar?

In [ ]:
if judge_df is None:
    print("No judge scores found in", JUDGE_DIR)
    print("Run experiments/run_judge_evaluation.py first.")
else:
    score_cols = [f"{d}_score" for d in JUDGE_DIMS]
    means = judge_df.groupby("condition")[score_cols].mean()

    # Radar chart
    N = len(JUDGE_DIMS)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    for cond in means.index:
        vals = means.loc[cond].tolist() + [means.loc[cond].iloc[0]]
        ax.plot(angles, vals, color=COLORS.get(cond, "gray"),
                linewidth=2, label=cond.capitalize())
        ax.fill(angles, vals, color=COLORS.get(cond, "gray"), alpha=0.1)
    ax.set_thetagrids(np.degrees(angles[:-1]), JUDGE_LABELS)
    ax.set_ylim(1, 5)
    ax.set_yticks([2, 3, 4, 5])
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
    ax.set_title("Judge Quality: Mean Score by Condition", fontweight="bold", pad=20)
    plt.tight_layout()
    plt.show()

    # Summary table
    display(means.rename(columns=dict(zip(score_cols, JUDGE_LABELS))).round(2))

In [ ]:
if judge_df is not None:
    score_cols = [f"{d}_score" for d in JUDGE_DIMS]
    pivot = judge_df.pivot_table(
        index=["name", "condition"], values=score_cols, aggfunc="mean"
    )
    pivot.columns = JUDGE_LABELS

    fig, ax = plt.subplots(figsize=(12, max(6, len(pivot) * 0.45)))
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn",
                vmin=1, vmax=5, linewidths=0.4, ax=ax)
    ax.set_title("Judge Scores: Per Startup × Condition × Dimension", fontweight="bold")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()

---
## Section 4: Misclassification Analysis

**Answers:** Are failure modes systematic (sector-specific) or random? Do conditions fail on the same startups?

In [ ]:
for cond, df in dfs.items():
    if df is None:
        continue
    df_err = label_errors(df)
    fps = df_err[df_err["error_type"] == "FP"]
    fns = df_err[df_err["error_type"] == "FN"]
    print(f"\n{'='*70}")
    print(f"[{cond.upper()}]  TP={len(df_err[df_err.error_type=='TP'])}  "
          f"TN={len(df_err[df_err.error_type=='TN'])}  "
          f"FP={len(fps)}  FN={len(fns)}")
    for label, subset in [("FALSE POSITIVES (predicted INVEST, GT=PASS)", fps),
                          ("FALSE NEGATIVES (predicted PASS, GT=INVEST)", fns)]:
        print(f"  {label}:")
        if subset.empty:
            print("    (none)")
        for _, r in subset.iterrows():
            prob = r.get("probability_float")
            prob_str = f"{prob:.0%}" if prob is not None else "?"
            print(f"    {str(r.get('name',''))[:38]:<40} "
                  f"[{str(r.get('category_code','?')):<12}] prob={prob_str}")
            print(f"      {str(r.get('reasoning',''))[:120]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, error_type, title in [
    (axes[0], "FP", "False Positives by Sector"),
    (axes[1], "FN", "False Negatives by Sector"),
]:
    sector_counts = {}
    for cond, df in dfs.items():
        if df is None:
            continue
        df_err = label_errors(df)
        subset = df_err[df_err["error_type"] == error_type]
        sector_counts[cond] = subset["category_code"].value_counts().head(8)

    all_sectors = sorted(set().union(*[set(v.index) for v in sector_counts.values()]))
    if not all_sectors:
        ax.text(0.5, 0.5, f"No {error_type}s", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title, fontweight="bold")
        continue

    x = np.arange(len(all_sectors))
    w = 0.25
    for i, (cond, counts) in enumerate(sector_counts.items()):
        vals = [counts.get(s, 0) for s in all_sectors]
        ax.bar(x + i * w, vals, w, label=cond,
               color=COLORS.get(cond, "gray"), alpha=0.85)
    ax.set_xticks(x + w)
    ax.set_xticklabels(all_sectors, rotation=40, ha="right", fontsize=9)
    ax.set_ylabel("Count")
    ax.set_title(title, fontweight="bold")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Startups misclassified by ALL conditions — structural hard cases
all_error_ids: dict[str, set] = {}
for cond, df in dfs.items():
    if df is None:
        continue
    df_err = label_errors(df)
    all_error_ids[cond] = set(df_err[df_err["error_type"].isin(["FP", "FN"])]["object_id"])

if all_error_ids:
    unanimous_errors = set.intersection(*all_error_ids.values()) if all_error_ids else set()
    print(f"Startups misclassified by ALL conditions ({len(unanimous_errors)}):")
    ref_df = next(df for df in dfs.values() if df is not None)
    for oid in sorted(unanimous_errors):
        row = ref_df[ref_df["object_id"] == oid]
        if not row.empty:
            r = row.iloc[0]
            gt = "INVEST" if r["target"] == 1 else "PASS"
            print(f"  {str(r.get('name',''))[:40]:<42} [{r.get('category_code','?')}]  GT={gt}")

---
## Section 5: Sector Performance

AUROC, balanced accuracy, and AP@K broken down by startup sector.

> **Requires re-running ablation experiments** after the `compute_sector_metrics` fix in `experiments/run_ablation.py`.
> Sectors with fewer than 3 examples are excluded.

**Answers:** Does TextGrad optimization generalize across sectors or overfit to the training distribution?

In [ ]:
# Startup count and base rate per sector — always available from predictions
ref_df = next((df for df in dfs.values() if df is not None), None)
if ref_df is not None:
    sector_summary = (
        ref_df.groupby("category_code")
        .agg(n=("target", "count"), n_success=("target", "sum"))
        .assign(base_rate=lambda x: (x["n_success"] / x["n"]).round(3))
        .sort_values("n", ascending=False)
    )
    print("Startup distribution by sector (from predictions):")
    display(sector_summary)
else:
    print("No prediction data available.")

In [ ]:
sector_data = {c: load_sector_metrics(c) for c in CONDITIONS}
available = {c: v for c, v in sector_data.items() if v}

if not available:
    print("No sector_metrics.json files found.")
    print("Re-run experiments/run_ablation.py after adding compute_sector_metrics (B2 fix).")
else:
    for metric_key, metric_label in [("auroc", "AUROC"), ("ap_10", "AP@10"), ("balanced_accuracy", "Balanced Accuracy")]:
        all_sectors = sorted(set().union(*[set(v.keys()) for v in available.values()]))
        matrix = pd.DataFrame(index=all_sectors, columns=list(available.keys()), dtype=float)
        for cond, data in available.items():
            for sector in all_sectors:
                matrix.loc[sector, cond] = data.get(sector, {}).get(metric_key, np.nan)

        fig, ax = plt.subplots(figsize=(8, max(4, len(all_sectors) * 0.5)))
        sns.heatmap(matrix.astype(float), annot=True, fmt=".2f",
                    cmap="RdYlGn", vmin=0, vmax=1, linewidths=0.4, ax=ax)
        ax.set_title(f"Sector-Level {metric_label} by Condition", fontweight="bold")
        plt.tight_layout()
        plt.show()
    display(matrix.round(3))

---
## Section 6: TextGrad Gradient Analysis

### 6a. Textual Gradients

> **Full gradient text requires re-running** `experiments/run_textgrad.py` after the B1 fix (`judge_feedback_full` field). Old runs show only the 100-char excerpt.

**Answers:** What does the backward model actually tell the synthesizer to do differently across steps?

In [ ]:
metrics_path = TG_DIR / "metrics_per_step.jsonl"
all_step_records = []
if metrics_path.exists():
    for line in metrics_path.read_text().splitlines():
        if line.strip():
            all_step_records.append(json.loads(line))

train_recs = [r for r in all_step_records if "predicted_decision" in r]
val_recs   = [r for r in all_step_records if "val_metrics" in r]

if not train_recs:
    print("No TextGrad training step records found at", metrics_path)
else:
    print(f"Found {len(train_recs)} training step(s), {len(val_recs)} validation checkpoint(s)\n")
    for rec in train_recs:
        step = rec.get("step", "?")
        name = rec.get("training_example_name", "?")
        gt   = rec.get("ground_truth", "?")
        pred = rec.get("predicted_decision", "?")
        ok   = rec.get("prediction_correct", "?")
        feedback = rec.get("judge_feedback_full") or rec.get("judge_feedback_excerpt", "")
        has_full = "judge_feedback_full" in rec
        print(f"{'─'*70}")
        print(f"Step {step}  |  {name}  |  GT={gt}  Pred={pred}  Correct={ok}")
        print(f"{'[FULL GRADIENT]' if has_full else '[EXCERPT ONLY — re-run for full text]'}")
        print(feedback[:800] if feedback else "(no feedback saved)")
        print()

In [ ]:
if train_recs:
    df_train = pd.DataFrame(train_recs)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Prompt length change per step, colored by prediction correctness
    if "prompt_length_change" in df_train.columns:
        bar_colors = [
            "#2ecc71" if c else "#e74c3c"
            for c in df_train.get("prediction_correct", pd.Series([True] * len(df_train)))
        ]
        axes[0].bar(df_train["step"], df_train["prompt_length_change"], color=bar_colors)
        axes[0].axhline(0, color="black", lw=0.8)
        axes[0].set_xlabel("Training step")
        axes[0].set_ylabel("Prompt length change (chars)")
        axes[0].set_title("Prompt Growth per Step\n(green=correct pred, red=wrong pred)")
        axes[0].legend(
            handles=[mpatches.Patch(color="#2ecc71", label="Correct"),
                     mpatches.Patch(color="#e74c3c", label="Wrong")]
        )

    # Validation metrics over steps
    if val_recs:
        df_val = pd.DataFrame([{**r, **r["val_metrics"]} for r in val_recs])
        for metric, label, color in [
            ("auroc", "AUROC", "#3498db"),
            ("balanced_accuracy", "Bal. Acc.", "#e67e22"),
            ("ap_10", "AP@10", "#9b59b6"),
        ]:
            if metric in df_val.columns:
                axes[1].plot(df_val["step"], df_val[metric], marker="o",
                             color=color, linewidth=2, label=label)
        axes[1].set_ylim(0, 1.05)
        axes[1].set_xlabel("Training step")
        axes[1].set_ylabel("Score")
        axes[1].set_title("Validation Metrics per Step")
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, "No validation checkpoints found",
                     ha="center", va="center", transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()